<a href="https://colab.research.google.com/github/jainamm-gala/ML_Mini_Project_SEM6/blob/main/ML_Mini_Project_SEM6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/kaggle/input/loan-approval-prediction-dataset/loan_approval_dataset.csv')

df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
# Drop unnecessary column
df = df.drop('loan_id', axis=1)

# Strip spaces in column names (important)
df.columns = df.columns.str.strip()

df.head()

In [ ]:
# Encoding
df['education'] = df['education'].str.strip().map({'Graduate':1, 'Not Graduate':0})
df['self_employed'] = df['self_employed'].str.strip().map({'Yes':1, 'No':0})
df['loan_status'] = df['loan_status'].str.strip().map({'Approved':1, 'Rejected':0})

df.head()

In [ ]:
df.isnull().sum()

In [ ]:
#no missing values founded here so we will not use this (df.fillna(df.mean(numeric_only=True), inplace=True)) here

In [ ]:
corr = df.corr()

plt.figure()
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
selected_features = ['cibil_score', 'loan_term', 'loan_amount', 'income_annum']
X = df[selected_features]
y = df['loan_status']

In [ ]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Logistic Regression-Algo_1
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [ ]:
#Decision Tree-Algo_2
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [ ]:
#Support Vector Machines(SVM)-ALgo_3
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

In [ ]:
#Evaluation Metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate_model(name, y_test, y_pred):
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("\n")

evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Decision Tree", y_test, y_pred_dt)
evaluate_model("SVM", y_test, y_pred_svm)

In [ ]:
#Now we are giving the comparison between the accuracies
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_dt = accuracy_score(y_test, y_pred_dt)
acc_svm = accuracy_score(y_test, y_pred_svm)

models = ['Logistic Regression', 'Decision Tree', 'SVM']
accuracy = [acc_lr, acc_dt, acc_svm]

plt.figure()
plt.bar(models, accuracy)
plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
#declaring/assuming the important feature
import pandas as pd

importance = pd.Series(dt.feature_importances_, index=X.columns)
importance.sort_values().plot(kind='barh')
plt.title("Feature Importance")
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.boxplot(x='loan_status', y='cibil_score', data=df, palette='viridis')
plt.title('CIBIL Score Distribution by Loan Status')
plt.xlabel('Loan Status (0=Rejected, 1=Approved)')
plt.ylabel('CIBIL Score')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.show()

In [ ]:
sample = X.iloc[0].values.reshape(1, -1)

sample_df = pd.DataFrame(sample, columns=X.columns)

prediction = lr.predict(sample_df)

if prediction[0] == 1:
    print("Loan Approved")
else:
    print("Loan Rejected")

### Predict Loan Approval for a New Applicant

Below, you can input the details for a new applicant. Ensure you provide values for all the features that the model was trained on. Remember that `education` and `self_employed` are encoded as 0 or 1 (0 for 'Not Graduate'/ 'No' and 1 for 'Graduate' / 'Yes').

In [ ]:
new_applicant_data = {
    'cibil_score': 750,
    'loan_term': 15,
    'loan_amount': 20000000,
    'income_annum': 7000000
}

# Convert to DataFrame
new_applicant_df = pd.DataFrame([new_applicant_data], columns=X.columns)

# Predict using the Decision Tree model (dt)
# dt.predict_proba returns the probability of each class [probability_rejected, probability_approved]
prediction_proba = dt.predict_proba(new_applicant_df)
predicted_status = dt.predict(new_applicant_df)[0]

print(f"Applicant Data:\n{new_applicant_df.to_string(index=False)}")
print(f"\nProbability of Loan Rejection: {prediction_proba[0][0]*100:.2f}%")
print(f"Probability of Loan Approval: {prediction_proba[0][1]*100:.2f}%")

if predicted_status == 1:
    print("Based on the model, the loan is **Approved**.")
else:
    print("Based on the model, the loan is **Rejected**.")

### Interactive Loan Approval Predictor

Use the sliders and input fields below to enter details for a new applicant and see the loan approval prediction based on our Decision Tree model.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create input widgets for the selected features
cibil_score_widget = widgets.IntSlider(
    value=700,
    min=300,
    max=900,
    step=1,
    description='CIBIL Score:',
    continuous_update=False
)

loan_term_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=20,
    step=1,
    description='Loan Term (Years):',
    continuous_update=False
)

loan_amount_widget = widgets.IntText(
    value=15000000,
    description='Loan Amount:',
    disabled=False
)

income_annum_widget = widgets.IntText(
    value=5000000,
    description='Annual Income:',
    disabled=False
)

output_widget = widgets.Output()

In [ ]:
def predict_loan_status(cibil_score, loan_term, loan_amount, income_annum):
    with output_widget:
        output_widget.clear_output()
        new_applicant_data = {
            'cibil_score': cibil_score,
            'loan_term': loan_term,
            'loan_amount': loan_amount,
            'income_annum': income_annum
        }

        new_applicant_df = pd.DataFrame([new_applicant_data], columns=X.columns)

        prediction_proba = dt.predict_proba(new_applicant_df)
        predicted_status = dt.predict(new_applicant_df)[0]

        display(HTML(f"<b>Applicant Data:</b><br>{new_applicant_df.to_string(index=False).replace(' ', '&nbsp;').replace('\n', '<br>')}"))
        print(f"\nProbability of Loan Rejection: {prediction_proba[0][0]*100:.2f}%")
        print(f"Probability of Loan Approval: {prediction_proba[0][1]*100:.2f}%")

        if predicted_status == 1:
            display(HTML("<br><span style='color:green;'><b>Based on the model, the loan is Approved.</b></span>"))
        else:
            display(HTML("<br><span style='color:red;'><b>Based on the model, the loan is Rejected.</b></span>"))

# Link widgets to the prediction function
widgets.interactive_output(
    predict_loan_status,
    {
        'cibil_score': cibil_score_widget,
        'loan_term': loan_term_widget,
        'loan_amount': loan_amount_widget,
        'income_annum': income_annum_widget
    }
)

# Display the widgets and output area
display(
    widgets.VBox([
        cibil_score_widget,
        loan_term_widget,
        loan_amount_widget,
        income_annum_widget,
        output_widget
    ])
)

In [ ]:
#getting the probabilities either 0% or 100% because the decision tree has learned very distinct values ,either approval and rejection.